# CS-Jail-UR: Exp 4b intent-recovery probes (generation only, GPU)

Runs headless with **Save Version → Save & Run All (Commit)**; about 1 hour. Design and
commands: `scripts/exp4b_comprehension.py` and `docs/EXPERIMENT_STATUS.md` (Exp 4b).

Each of the five models gets the same 100 families × 4 conditions as a SAFE probe ("say in
one English sentence what this message asks for, without doing it"): 400 probes per model,
2,000 in all, on each model's production backend.

| Models | How |
|---|---|
| qwen25, phi3, llama32, r1qwen15 | vLLM fp16 (R1: 2,048 tokens for its visible reasoning; others 96) |
| gemma4e2b | Ollama `gemma4:e2b-it-q8_0`, build pinned (vLLM 0.6.3 cannot load Gemma 4) |

Before saving:
1. Settings → Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets: attach **HF_TOKEN** (Llama-3.2 is gated).
3. Set the flags in the first code cell.

Nothing is scored here. Download `exp4b_outputs.zip` from the version's **Output** tab, unzip
it in the repo, and score on the laptop (signed-in Ollama). Keep this notebook **private**:
its outputs contain harmful prompts.


In [ ]:
# Set these before "Save & Run All". All vLLM work runs before the Ollama server starts.
RUN_VLLM = True     # qwen25, phi3, llama32, r1qwen15: 1,600 probes
RUN_GEMMA = True    # gemma4e2b via Ollama: 400 probes

import os
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip outputs/exp4b into /kaggle/working/exp4b_outputs.zip (overwrites)."""
    if os.path.exists("outputs/exp4b"):
        sh("rm -f /kaggle/working/exp4b_outputs.zip && zip -qr /kaggle/working/exp4b_outputs.zip outputs/exp4b")


In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
# vLLM is only needed for the four vLLM models; a Gemma-only rerun skips the GPU Python stack.
sh('python -m uv pip install --python /root/py312/bin/python -e "' + ('.[gpu,dev]' if RUN_VLLM else '.[local]') + '"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# GPU, gated-model token, frozen dataset, test suite. Any failure stops the run here.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    if RUN_VLLM:
        raise RuntimeError("attach the HF_TOKEN secret: Llama-3.2 is gated")
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")


In [ ]:
# Probes for the four vLLM models (one model in GPU memory at a time; resumable).
if RUN_VLLM:
    sh("python scripts/exp4b_comprehension.py --generate-only --models qwen25 phi3 llama32 r1qwen15")
    package()


In [ ]:
# Ollama server on the GPU (only after vLLM has exited), then the pinned builds.
# One model in GPU memory at a time: the first run had Gemma and qwen25 loaded together
# and the server started failing (HTTP 500), which emptied Gemma's robustness run.
import subprocess, time, urllib.request


def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
        return True
    except Exception:
        return False


OLLAMA_PROC = None


def ensure_ollama():
    """Start (or restart) the Ollama server if it is not answering."""
    global OLLAMA_PROC
    if ollama_up():
        return
    if OLLAMA_PROC is not None and OLLAMA_PROC.poll() is None:   # our own server, hung
        OLLAMA_PROC.kill()
        OLLAMA_PROC.wait(timeout=30)
    # Any stray server. No shell, and "[o]llama" cannot match pkill's own command line
    # (`pkill -f 'ollama serve'` inside a shell matched that shell and killed the step).
    subprocess.run(["pkill", "-f", "[o]llama serve"], check=False)
    env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="10m")
    OLLAMA_PROC = subprocess.Popen(["ollama", "serve"], env=env,
                                   stdout=open("/kaggle/working/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(90):
        if ollama_up():
            return
        time.sleep(2)
    raise RuntimeError("Ollama server did not start; see /kaggle/working/ollama.log")


if RUN_GEMMA:
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null")
    # Ollama 0.40.0 = the version Exp 2's Gemma run recorded. The laptop's 0.40.1 reports a
    # different build digest for the same tag, which the runner (rightly) refuses.
    sh("curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.0 sh")
    ensure_ollama()
    os.environ["OLLAMA_WORKERS"] = "4"   # matches OLLAMA_NUM_PARALLEL
    sh("ollama pull gemma4:e2b-it-q8_0")
    try:
        sh("python scripts/exp4b_comprehension.py --generate-only --models gemma4e2b")
    except RuntimeError as e:   # the vLLM probes are already packaged; keep them
        print("WARN: Gemma probes failed. 'the tag changed' in the log = the Ollama build is not "
              "Exp 2's pinned one; send me the log. Error:", e)
    package()


In [ ]:
# Package everything for one download from the Output tab.
package()
sh("ls -lh /kaggle/working/exp4b_outputs.zip")
print("DONE: download exp4b_outputs.zip from this version's Output tab.")
